In [31]:
import os, sys
 
WORK  = '/kaggle/working'
REPO  = 'https://github.com/PhuongThao-2005/TULIP-MedML.git'
RNAME = 'TULIP-MedML'
 
if os.path.exists(f'{WORK}/{RNAME}'):
    os.system(f'cd {WORK}/{RNAME} && git pull')
    print('Pulled latest')
else:
    ret = os.system(f'cd {WORK} && git clone {REPO}')
    print('Cloned' if ret == 0 else 'Clone FAILED')
 
os.system('pip install pyyaml timm -q')
sys.path.insert(0, f'{WORK}/{RNAME}')
os.chdir(f'{WORK}/{RNAME}')
 
print('Working dir :', os.getcwd())
print('Repo contents:', os.listdir('.'))

Already up to date.
Pulled latest
Working dir : /kaggle/working/TULIP-MedML
Repo contents: ['data', 'src', '.gitignore', 'README.md', 'requirements.txt', '.git', 'notebooks']


In [32]:
%%writefile /kaggle/working/TULIP-MedML/src/engine.py

import os
import shutil
import time

import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim
import torch.utils.data
import torchvision.transforms as transforms

from src.util import Warp, MultiScaleCrop, AveragePrecisionMeter

from src.evaluate import (
    compute_AUC_uncertain,
    compute_mAP,
    compute_mean_AUC,
    print_metrics,
)


# ─── Running average ─────────────────────────────────────────────────────────

class _AverageMeter:
    """Running average — tracks loss across batches within one epoch."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.sum = 0.0
        self.count = 0
        self.avg = 0.0

    def add(self, val: float, n: int = 1):
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

    def value(self):
        return (self.avg, 0.0)


# ─── Base Engine ─────────────────────────────────────────────────────────────

class Engine:
    """
    Base training loop using a callback/hook pattern.

    Subclasses override hooks rather than copy the full loop:
        on_start_epoch → on_start_batch → on_forward → on_end_batch → on_end_epoch

    GCNMultiLabelMAPEngine overrides on_start_batch (unpack tuple + remap
    labels) and on_forward (single-input GCN forward pass — inp is a model
    buffer, not a dataset field).
    """

    def __init__(self, state: dict = {}):
        self.state = state

        defaults = {
            'use_gpu'     : torch.cuda.is_available(),
            'image_size'  : 224,
            'batch_size'  : 64,
            'workers'     : 4,
            'device_ids'  : None,   # None → use all GPUs
            'evaluate'    : False,  # True → validate only, no training
            'start_epoch' : 0,
            'max_epochs'  : 90,
            'epoch_step'  : [],     # epochs at which to decay lr × 0.1
            'use_pb'      : True,
            'print_freq'  : 0,
            # 'loss_type' : 'bce'  ← caller sets this via state dict
        }
        for k, v in defaults.items():
            if self._state(k) is None:
                self.state[k] = v

        self.state['meter_loss'] = _AverageMeter()
        self.state['batch_time'] = _AverageMeter()
        self.state['data_time']  = _AverageMeter()

    def _state(self, name):
        return self.state.get(name)

    # ── Hooks ────────────────────────────────────────────────────────────────

    def on_start_epoch(self, training, model, criterion, data_loader, optimizer=None, display=True):
        self.state['meter_loss'].reset()
        self.state['batch_time'].reset()
        self.state['data_time'].reset()

    def on_end_epoch(self, training, model, criterion, data_loader, optimizer=None, display=True):
        loss = self.state['meter_loss'].avg
        if display:
            tag = 'Epoch: [{0}]'.format(self.state['epoch']) if training else 'Test:'
            print(f'{tag}\tLoss {loss:.4f}')
        return loss

    def on_start_batch(self, training, model, criterion, data_loader, optimizer=None, display=True):
        pass

    def on_end_batch(self, training, model, criterion, data_loader, optimizer=None, display=True):
        self.state['loss_batch'] = self.state['loss'].item()
        self.state['meter_loss'].add(self.state['loss_batch'])

        if display and self.state['print_freq'] != 0 \
                and self.state['iteration'] % self.state['print_freq'] == 0:
            loss = self.state['meter_loss'].avg
            if training:
                print('Epoch: [{0}][{1}/{2}]\t'
                      'Time {btc:.3f} ({bt:.3f})\t'
                      'Data {dtc:.3f} ({dt:.3f})\t'
                      'Loss {lc:.4f} ({l:.4f})'.format(
                    self.state['epoch'], self.state['iteration'], len(data_loader),
                    btc=self.state['batch_time_current'],
                    bt=self.state['batch_time'].avg,
                    dtc=self.state['data_time_batch'],
                    dt=self.state['data_time'].avg,
                    lc=self.state['loss_batch'], l=loss))
            else:
                print('Test: [{0}/{1}]\t'
                      'Time {btc:.3f} ({bt:.3f})\t'
                      'Data {dtc:.3f} ({dt:.3f})\t'
                      'Loss {lc:.4f} ({l:.4f})'.format(
                    self.state['iteration'], len(data_loader),
                    btc=self.state['batch_time_current'],
                    bt=self.state['batch_time'].avg,
                    dtc=self.state['data_time_batch'],
                    dt=self.state['data_time'].avg,
                    lc=self.state['loss_batch'], l=loss))

    def on_forward(self, training, model, criterion, data_loader,
                   optimizer=None, display=True):
        input_var  = self.state['input']
        target_var = self.state['target']

        self.state['output'] = model(input_var)
        self.state['loss']   = criterion(self.state['output'], target_var)

        if training:
            optimizer.zero_grad()
            self.state['loss'].backward()
            optimizer.step()

    # ── Transforms ───────────────────────────────────────────────────────────

    def init_learning(self, model, criterion):
        """
        Tạo image transforms nếu chưa có.
        Gọi mean/std từ model thay vì hardcode → mỗi backbone có thể dùng
        chuẩn hóa khác nhau (ImageNet, BiomedCLIP, etc.)
        """
        mean = getattr(model, 'image_normalization_mean', [0.485, 0.456, 0.406])
        std  = getattr(model, 'image_normalization_std',  [0.229, 0.224, 0.225])
        normalize = transforms.Normalize(mean=mean, std=std)

        if self._state('train_transform') is None:
            self.state['train_transform'] = transforms.Compose([
                MultiScaleCrop(self.state['image_size'],
                               scales=(1.0, 0.875, 0.75, 0.66, 0.5),
                               max_distort=2),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                normalize,
            ])

        if self._state('val_transform') is None:
            self.state['val_transform'] = transforms.Compose([
                Warp(self.state['image_size']),
                transforms.ToTensor(),
                normalize,
            ])

        self.state['best_score'] = 0.0

    # ── Main loop ────────────────────────────────────────────────────────────

    def learning(self, model, criterion, train_dataset, val_dataset, optimizer=None):
        self.init_learning(model, criterion)

        train_dataset.transform = self.state['train_transform']
        train_dataset.target_transform = self._state('train_target_transform')
        val_dataset.transform   = self.state['val_transform']
        val_dataset.target_transform   = self._state('val_target_transform')

        train_loader = torch.utils.data.DataLoader(
            train_dataset,
            batch_size=self.state['batch_size'],
            shuffle=True,
            num_workers=self.state['workers'],
            pin_memory=self.state['use_gpu'],
        )
        val_loader = torch.utils.data.DataLoader(
            val_dataset,
            batch_size=self.state['batch_size'],
            shuffle=False,
            num_workers=self.state['workers'],
            pin_memory=self.state['use_gpu'],
        )

        if self.state['use_gpu']:
            model = torch.nn.DataParallel(
                model, device_ids=self.state['device_ids']).cuda()

        # Resume từ checkpoint nếu có
        if self._state('resume') is not None:
            if os.path.isfile(self.state['resume']):
                print("=> loading checkpoint '{}'".format(self.state['resume']))
                checkpoint = torch.load(self.state['resume'], map_location='cpu')
                self.state['start_epoch'] = checkpoint['epoch'] + 1
                self.state['best_score']  = checkpoint['best_score']
                model.load_state_dict(checkpoint['state_dict'])
                if optimizer is not None and 'optimizer' in checkpoint:
                    optimizer.load_state_dict(checkpoint['optimizer'])
                print("=> loaded checkpoint (epoch {})".format(checkpoint['epoch']))
            else:
                print("=> no checkpoint found at '{}'".format(self.state['resume']))

        if self.state['evaluate']:
            self.validate(val_loader, model, criterion)
            return

        for epoch in range(self.state['start_epoch'], self.state['max_epochs']):
            self.state['epoch'] = epoch
            lr = self.adjust_learning_rate(optimizer)
            print('lr:', lr)

            self.train(train_loader, model, criterion, optimizer, epoch)
            score = self.validate(val_loader, model, criterion)

            # Lưu checkpoint mỗi epoch, đánh dấu is_best nếu score tốt nhất
            is_best = score > self.state['best_score']
            self.state['best_score'] = max(score, self.state['best_score'])
            self.save_checkpoint({
                'epoch'     : epoch,
                'arch'      : self._state('arch'),
                'state_dict': model.module.state_dict()
                               if hasattr(model, 'module') else model.state_dict(),
                'best_score': self.state['best_score'],
                'optimizer' : optimizer.state_dict(),
            }, is_best)
            print('best score:', self.state['best_score'])

        return self.state['best_score']

    def train(self, data_loader, model, criterion, optimizer, epoch):
        model.train()
        self.on_start_epoch(True, model, criterion, data_loader, optimizer)

        if self.state['use_pb']:
            data_loader = tqdm(data_loader, desc='Train')

        end = time.time()
        for i, (input, target) in enumerate(data_loader):
            self.state['iteration']       = i
            self.state['data_time_batch'] = time.time() - end
            self.state['data_time'].add(self.state['data_time_batch'])

            self.state['input']  = input
            self.state['target'] = target

            self.on_start_batch(True, model, criterion, data_loader, optimizer)

            if self.state['use_gpu']:
                self.state['target'] = self.state['target'].cuda(non_blocking=True)

            self.on_forward(True, model, criterion, data_loader, optimizer)

            self.state['batch_time_current'] = time.time() - end
            self.state['batch_time'].add(self.state['batch_time_current'])
            end = time.time()

            self.on_end_batch(True, model, criterion, data_loader, optimizer)

        self.on_end_epoch(True, model, criterion, data_loader, optimizer)

    def validate(self, data_loader, model, criterion):
        model.eval()
        self.on_start_epoch(False, model, criterion, data_loader)

        if self.state['use_pb']:
            data_loader = tqdm(data_loader, desc='Val')

        end = time.time()
        with torch.no_grad():
            for i, (input, target) in enumerate(data_loader):
                self.state['iteration']       = i
                self.state['data_time_batch'] = time.time() - end
                self.state['data_time'].add(self.state['data_time_batch'])

                self.state['input']  = input
                self.state['target'] = target

                self.on_start_batch(False, model, criterion, data_loader)

                if self.state['use_gpu']:
                    self.state['target'] = self.state['target'].cuda(non_blocking=True)

                self.on_forward(False, model, criterion, data_loader)

                self.state['batch_time_current'] = time.time() - end
                self.state['batch_time'].add(self.state['batch_time_current'])
                end = time.time()

                self.on_end_batch(False, model, criterion, data_loader)

        return self.on_end_epoch(False, model, criterion, data_loader)

    def save_checkpoint(self, state, is_best, filename=None):
        if filename is None:
            filename = f'checkpoint_epoch_{state["epoch"]}.pth.tar'
        if self._state('save_model_path') is not None:
            filename_ = filename
            filename  = os.path.join(self.state['save_model_path'], filename_)
            os.makedirs(self.state['save_model_path'], exist_ok=True)
        print('save model {filename}'.format(filename=filename))
        torch.save(state, filename)

        if is_best:
            filename_best = 'model_best.pth.tar'
            if self._state('save_model_path') is not None:
                filename_best = os.path.join(self.state['save_model_path'],
                                             filename_best)
            shutil.copyfile(filename, filename_best)

            if self._state('save_model_path') is not None:
                if self._state('filename_previous_best') is not None:
                    try:
                        os.remove(self._state('filename_previous_best'))
                    except OSError:
                        pass
                filename_best = os.path.join(
                    self.state['save_model_path'],
                    'model_best_{score:.4f}.pth.tar'.format(
                        score=state['best_score']))
                shutil.copyfile(filename, filename_best)
                self.state['filename_previous_best'] = filename_best

    def adjust_learning_rate(self, optimizer):
        """
        Decay lr × 0.1 tại mỗi epoch trong epoch_step.
        Tính từ base_lrs gốc thay vì *= để tránh lỗi khi resume.
        """
        # Lưu lr gốc lần đầu tiên gọi
        if 'base_lrs' not in self.state:
            self.state['base_lrs'] = [pg['lr'] for pg in optimizer.param_groups]

        n_decays = sum(
            1 for s in self.state['epoch_step']
            if s <= self.state['epoch']
        )
        factor = 0.1 ** n_decays

        lr_list = []
        for pg, base_lr in zip(optimizer.param_groups, self.state['base_lrs']):
            pg['lr'] = base_lr * factor
            lr_list.append(pg['lr'])
        return np.unique(lr_list)


# ─── Multi-label mAP Engine ──────────────────────────────────────────────────

class MultiLabelMAPEngine(Engine):
    """
    Extends Engine với mAP metric cho multi-label classification.
    """

    def __init__(self, state: dict):
        Engine.__init__(self, state)
        if self._state('difficult_examples') is None:
            self.state['difficult_examples'] = False
        self.state['ap_meter'] = AveragePrecisionMeter(
            self.state['difficult_examples'])

    def on_start_epoch(self, training, model, criterion, data_loader,
                       optimizer=None, display=True):
        Engine.on_start_epoch(self, training, model, criterion, data_loader,
                              optimizer)
        self.state['ap_meter'].reset()

    def on_end_epoch(self, training, model, criterion, data_loader,
                     optimizer=None, display=True):
        map_val = 100.0 * self.state['ap_meter'].value().mean()
        loss    = self.state['meter_loss'].avg
        if display:
            tag = ('Epoch: [{0}]'.format(self.state['epoch'])
                   if training else 'Test:')
            print(f'{tag}\tLoss {loss:.4f}\tmAP {map_val:.3f}')
        return map_val

    def on_start_batch(self, training, model, criterion, data_loader,
                       optimizer=None, display=True):
        self.state['target_gt'] = self.state['target'].clone()

        input = self.state['input']
        self.state['input'] = input[0]
        self.state['name']  = input[1]

    def on_end_batch(self, training, model, criterion, data_loader,
                     optimizer=None, display=True):
        Engine.on_end_batch(self, training, model, criterion, data_loader,
                            display=False)
        self.state['ap_meter'].add(
            self.state['output'].data,
            self.state['target_gt'])

        if display and self.state['print_freq'] != 0 \
                and self.state['iteration'] % self.state['print_freq'] == 0:
            loss = self.state['meter_loss'].avg
            if training:
                print('Epoch: [{0}][{1}/{2}]\t'
                      'Loss {lc:.4f} ({l:.4f})'.format(
                    self.state['epoch'], self.state['iteration'],
                    len(data_loader),
                    lc=self.state['loss_batch'], l=loss))
            else:
                print('Test: [{0}/{1}]\tLoss {lc:.4f} ({l:.4f})'.format(
                    self.state['iteration'], len(data_loader),
                    lc=self.state['loss_batch'], l=loss))


# ─── GCN Engine ──────────────────────────────────────────────────────────────

class GCNMultiLabelMAPEngine(MultiLabelMAPEngine):
    """
    Extends MultiLabelMAPEngine cho GCNResnet.

    Thay đổi chính so với phiên bản cũ:
      1. inp KHÔNG còn trong dataset — model tự load qua register_buffer.
         on_start_batch chỉ unpack (img, path), không cần lấy inp từ input[2].
      2. on_start_batch có 2 nhánh remap label tùy loss_type:
         - 'bce'    : -1 → 0  (uncertain coi như negative cho loss)
         - 'ua_asl' : giữ nguyên -1 (loss tự xử lý)
      3. on_forward gọi model(feature) một argument — inp là buffer trong model.
      4. Checkpoint chọn theo mAP (khớp proposal), fallback mean_auc.
      5. adjust_learning_rate tính từ base_lrs gốc, không lỗi khi resume.
    """

    def on_start_epoch(self, training, model, criterion, data_loader,
                       optimizer=None, display=True):
        MultiLabelMAPEngine.on_start_epoch(
            self, training, model, criterion, data_loader, optimizer)
        if not training:
            self.state['_val_scores']  = []
            self.state['_val_targets'] = []

    def on_start_batch(self, training, model, criterion, data_loader,
                       optimizer=None, display=True):
        # Preserve original labels for evaluation (keeps -1 uncertain)
        self.state['target_gt'] = self.state['target'].clone()

        loss_type = self.state.get('loss_type', 'bce')

        if loss_type == 'bce':
            # BCE không hiểu -1 → remap uncertain thành negative
            target = self.state['target'].clone().float()
            target[target < 0] = 0.0
            self.state['target'] = target
        else:
            # ua_asl nhận -1 trực tiếp, chỉ cast float
            self.state['target'] = self.state['target'].float()

        # Unpack (img, path)
        input = self.state['input']
        self.state['feature'] = input[0]   # [B, 3, H, W]
        self.state['out']     = input[1]   # list of path strings

    def on_end_batch(self, training, model, criterion, data_loader,
                     optimizer=None, display=True):
        MultiLabelMAPEngine.on_end_batch(
            self, training, model, criterion, data_loader, display)

        if not training:
            probs = torch.sigmoid(self.state['output']).detach().cpu().numpy()
            # Dùng target_gt (giữ -1) để compute_AUC_uncertain hoạt động đúng
            gt    = self.state['target_gt'].detach().cpu().numpy()
            self.state['_val_scores'].append(probs)
            self.state['_val_targets'].append(gt)

    def on_end_epoch(self, training, model, criterion, data_loader,
                     optimizer=None, display=True):
        loss = self.state['meter_loss'].avg

        if training:
            map_val = 100.0 * self.state['ap_meter'].value().mean()
            if display:
                print(f'Epoch: [{self.state["epoch"]}]\t'
                      f'Loss {loss:.4f}\tmAP {map_val:.3f}')
            return map_val

        # Validation
        val_scores  = self.state.get('_val_scores',  [])
        val_targets = self.state.get('_val_targets', [])

        if not val_scores:
            print(f'Val:\tLoss {loss:.4f}  (no predictions collected)')
            return 0.0

        scores  = np.concatenate(val_scores,  axis=0)   # [N, 14]
        targets = np.concatenate(val_targets, axis=0)   # [N, 14] — có thể có -1

        mAP, per_class_ap = compute_mAP(scores, targets)
        mean_auc, per_cls = compute_mean_AUC(scores, targets)
        unc_auc          = compute_AUC_uncertain(scores, targets)

        results = {
            'map'          : round(mAP,      4) if not np.isnan(mAP)      else None,
            'mean_auc'     : round(mean_auc, 4) if not np.isnan(mean_auc) else None,
            # unc_auc = nan là bình thường khi val set không có -1 (CheXpert official val)
            'unc_auc'      : round(unc_auc,  4) if not np.isnan(unc_auc)  else None,
            'per_class_auc': per_cls,
            'per_class_ap' : per_class_ap,
        }

        if display:
            print(f'\nVal:\tLoss {loss:.4f}')
            loss_type = self.state.get('loss_type', 'bce')
            print_metrics(results, show_unc=(loss_type == 'ua_asl'))

        # Dùng mAP làm primary score để chọn checkpoint (khớp proposal).
        # Fallback mean_auc nếu mAP không tính được.
        score = mAP if not np.isnan(mAP) else (mean_auc if not np.isnan(mean_auc) else 0.0)
        return float(score)

    def on_forward(self, training, model, criterion, data_loader,
                   optimizer=None, display=True):
        feature_var = self.state['feature'].float()
        target_var  = self.state['target'].float()

        if self.state['use_gpu']:
            feature_var = feature_var.cuda()

        # inp là register_buffer trong model — tự .cuda() cùng model,
        # không cần truyền qua đây nữa
        self.state['output'] = model(feature_var)   # logits [B, 14]
        self.state['loss']   = criterion(self.state['output'], target_var)

        if training:
            optimizer.zero_grad()
            self.state['loss'].backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            optimizer.step()

Overwriting /kaggle/working/TULIP-MedML/src/engine.py


In [33]:
%%writefile /kaggle/working/TULIP-MedML/src/evaluate.py

import torch
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

from src.data.chexpert import CHEXPERT_CLASSES


def _unpack_batch(batch):
    if not isinstance(batch, (tuple, list)) or len(batch) != 2:
        raise ValueError('Expected batch format (input, target).')

    inputs, labels = batch
    imgs = None
    word_vecs = None

    if torch.is_tensor(inputs):
        imgs = inputs
    elif isinstance(inputs, (tuple, list)):
        if len(inputs) == 0:
            raise ValueError('Input tuple is empty.')
        imgs = inputs[0]
        if len(inputs) >= 3 and torch.is_tensor(inputs[2]):
            word_vecs = inputs[2]
        elif len(inputs) >= 2 and torch.is_tensor(inputs[1]):
            word_vecs = inputs[1]
    else:
        raise ValueError(f'Unsupported input type: {type(inputs)}')

    if not torch.is_tensor(labels):
        labels = torch.as_tensor(labels)

    return imgs, word_vecs, labels


# ─────────────────────────────────────────────────────────
# 1. mAP — exclude nhãn -1
# ─────────────────────────────────────────────────────────
def compute_mAP(
    scores : np.ndarray,
    targets: np.ndarray,
) -> tuple[float, dict]:
    """
    scores:  [N, 14] sigmoid output
    targets: [N, 14] {-1, 0, 1}
    Bỏ sample có nhãn -1, chỉ tính trên {0, 1}.

    Trả về (mean_ap, per_class_dict) — khớp với compute_mean_AUC.
    per_class_dict: {label_name: ap} để in bảng.
    """
    per_class = {}
    aps       = []

    for i, cls in enumerate(CHEXPERT_CLASSES):
        t = targets[:, i]
        s = scores[:, i]

        mask = t != -1                      # bỏ uncertain
        if mask.sum() == 0:
            continue
        t_bin = (t[mask] == 1).astype(int)
        if t_bin.sum() == 0:               # không có positive
            continue

        try:
            ap = average_precision_score(t_bin, s[mask])
            per_class[cls] = round(float(ap), 4)
            aps.append(ap)
        except ValueError:
            pass

    mean_ap = float(np.mean(aps)) if aps else float('nan')
    return mean_ap, per_class


# ─────────────────────────────────────────────────────────
# 2. mean AUC — bỏ nhãn không có dương tính
# ─────────────────────────────────────────────────────────
def compute_mean_AUC(
    scores : np.ndarray,
    targets: np.ndarray,
) -> tuple[float, dict]:
    """
    Trả về (mean_auc, per_class_dict).
    per_class_dict: {label_name: auc} để in bảng.
    """
    per_class = {}
    aucs      = []

    for i, cls in enumerate(CHEXPERT_CLASSES):
        t = targets[:, i]
        s = scores[:, i]

        mask = t != -1                      # bỏ uncertain
        if mask.sum() == 0:
            continue
        t_bin = (t[mask] == 1).astype(int)
        if t_bin.sum() == 0 or (1 - t_bin).sum() == 0:
            continue                        # cần cả pos lẫn neg

        try:
            auc = roc_auc_score(t_bin, s[mask])
            per_class[cls] = round(float(auc), 4)
            aucs.append(auc)
        except ValueError:
            pass

    mean_auc = float(np.mean(aucs)) if aucs else float('nan')
    return mean_auc, per_class


# ─────────────────────────────────────────────────────────
# 3. AUC uncertain — subset sample có ≥1 nhãn -1
# ─────────────────────────────────────────────────────────
def compute_AUC_uncertain(
    scores : np.ndarray,
    targets: np.ndarray,
) -> float:
    """
    Chỉ tính trên subset sample có ít nhất 1 nhãn = -1.
    Map uncertain (-1) → positive (1) khi evaluate.

    Lý do: ảnh uncertain thường có dấu hiệu bệnh mờ nhạt
    → model tốt phải cho score cao hơn ảnh âm tính rõ ràng
    → nếu UA-ASL hoạt động đúng, unc_auc sẽ cao hơn BCE.

    Lưu ý: CheXpert official val set chỉ có {0, 1} → luôn trả nan.
    Metric này chỉ có ý nghĩa khi eval trên train subset hoặc uncertain split.
    """
    # Lấy subset có ít nhất 1 nhãn -1
    has_uncertain = (targets == -1).any(axis=1)
    if has_uncertain.sum() == 0:
        return float('nan')

    s_unc = scores[has_uncertain]           # [M, 14]
    t_unc = targets[has_uncertain].copy()   # [M, 14]
    t_unc[t_unc == -1] = 1                  # uncertain → positive

    aucs = []
    for i in range(len(CHEXPERT_CLASSES)):
        t_c = t_unc[:, i]
        s_c = s_unc[:, i]
        if t_c.sum() == 0 or (1 - t_c).sum() == 0:
            continue
        try:
            auc = roc_auc_score(t_c, s_c)
            aucs.append(auc)
        except ValueError:
            pass

    return float(np.mean(aucs)) if aucs else float('nan')


# ─────────────────────────────────────────────────────────
# Interface chính
# ─────────────────────────────────────────────────────────
def evaluate(model, loader, device='cuda') -> dict:
    """
    Chạy inference toàn bộ loader, tính 3 metrics.

    Returns:
        {
            "map"          : float,
            "mean_auc"     : float,
            "unc_auc"      : float,
            "per_class_auc": dict,   # {label: auc}
            "per_class_ap" : dict,   # {label: ap}
        }
    """
    model.eval()

    all_scores  = []
    all_targets = []

    with torch.no_grad():
        for batch in loader:
            imgs, word_vecs, labels = _unpack_batch(batch)

            imgs      = imgs.to(device)
            labels    = labels.to(device)

            if word_vecs is not None:
                word_vecs = word_vecs.to(device)
                logits = model(imgs, word_vecs)     # [B, 14]
            else:
                logits = model(imgs)                # [B, 14]

            probs  = torch.sigmoid(logits)          # [B, 14]

            all_scores.append(probs.cpu().numpy())
            all_targets.append(labels.cpu().numpy())   # giữ nguyên {-1,0,1}

    if not all_scores:
        return {
            'map'          : None,
            'mean_auc'     : None,
            'unc_auc'      : None,
            'per_class_auc': {},
            'per_class_ap' : {},
        }

    scores  = np.concatenate(all_scores,  axis=0)  # [N, 14]
    targets = np.concatenate(all_targets, axis=0)  # [N, 14]

    map_score,  per_class_ap  = compute_mAP(scores, targets)
    mean_auc,   per_class_auc = compute_mean_AUC(scores, targets)
    unc_auc                   = compute_AUC_uncertain(scores, targets)

    return {
        'map'          : round(map_score, 4) if not np.isnan(map_score) else None,
        'mean_auc'     : round(mean_auc,  4) if not np.isnan(mean_auc)  else None,
        'unc_auc'      : round(unc_auc,   4) if not np.isnan(unc_auc)   else None,
        'per_class_auc': per_class_auc,
        'per_class_ap' : per_class_ap,
    }


# ─────────────────────────────────────────────────────────
# Print
# ─────────────────────────────────────────────────────────
def print_metrics(results: dict, show_unc: bool = False):
    """
    In bảng Class | AUC | AP per-class, rồi các dòng tổng.

    show_unc: chỉ in unc_auc khi dùng UA-ASL (C4/C5).
              Với BCE (C1/C2/C3) val set không có -1 → unc_auc luôn nan,
              không cần in.
    """
    per_auc = results.get('per_class_auc', {})
    per_ap  = results.get('per_class_ap',  {})

    print(f"\n{'Class':35s} {'AUC':>8} {'AP':>8}")
    print('-' * 55)

    for cls in CHEXPERT_CLASSES:
        auc = per_auc.get(cls, float('nan'))
        ap  = per_ap.get(cls,  float('nan'))

        auc_str = f'{auc:.4f}' if not np.isnan(auc) else 'nan'
        ap_str  = f'{ap:.4f}'  if not np.isnan(ap)  else 'nan'

        print(f'{cls:35s} {auc_str:>8} {ap_str:>8}')

    print('-' * 55)

    mean_auc = results.get('mean_auc')
    map_val  = results.get('map')
    unc      = results.get('unc_auc')

    mean_auc_str = 'nan' if mean_auc is None else f'{mean_auc:.4f}'
    map_str      = 'nan' if map_val  is None else f'{map_val:.4f}'

    # Gộp thành 1 dòng "Mean" đúng cột
    print(f"{'Mean':35s} {mean_auc_str:>8} {map_str:>8}")

    # unc_auc (nếu có) → để riêng (vì không thuộc cột AP)
    if show_unc:
        unc_str = 'nan' if unc is None else f'{unc:.4f}'
        print(f"{'unc_auc':35s} {unc_str:>8}")


Overwriting /kaggle/working/TULIP-MedML/src/evaluate.py


In [34]:
# ── Cell 2: Kiểm tra GPU + dataset ───────────────────────────────────
import torch, pandas as pd
 
print('GPU  :', torch.cuda.get_device_name(0))
print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
 
ROOT = '/kaggle/input/datasets/ashery/chexpert'
df   = pd.read_csv(f'{ROOT}/train.csv')
 
# Path trong CSV có dạng CheXpert-v1.0-small/... → bỏ phần đầu
sample = '/'.join(df['Path'].iloc[0].split('/')[1:])
abs_p  = os.path.join(ROOT, sample)
 
print('\nDataset root :', ROOT)
print('train.csv    :', len(df), 'rows')
print('val.csv      :', len(pd.read_csv(f'{ROOT}/valid.csv')), 'rows')
print('Image exists :', os.path.isfile(abs_p))
 
print('\nData files in repo:')
for fname in ['data/chexpert_glove_word2vec.npy', 'data/chexpert_adj.pkl']:
    exists = os.path.isfile(fname)
    size   = os.path.getsize(fname) / 1024 if exists else 0
    print(f'  {fname:45s} {"OK" if exists else "MISSING":8s} {size:.1f} KB')


GPU  : Tesla T4
VRAM : 15.6 GB

Dataset root : /kaggle/input/datasets/ashery/chexpert
train.csv    : 223414 rows
val.csv      : 234 rows
Image exists : True

Data files in repo:
  data/chexpert_glove_word2vec.npy              OK       16.5 KB
  data/chexpert_adj.pkl                         OK       1.0 KB


In [35]:
# ── Cell 3: Build adjacency matrix (nếu chưa có) ──────────────────────────────
# Co-occurrence được tính chỉ trên nhãn = 1 (positive).
# Uncertain (-1) và negative (0) đều không đếm là co-occur,
# để tránh inflate co-occurrence của các bệnh hiếm / không rõ.
import numpy as np, pickle
from src.data.chexpert import CHEXPERT_CLASSES
 
ADJ_PATH = 'data/chexpert_adj.pkl'
os.makedirs('data', exist_ok=True)
 
if not os.path.isfile(ADJ_PATH):
    print('Building adjacency matrix from train.csv ...')
    df_adj = pd.read_csv(f'{ROOT}/train.csv')
    labels = df_adj[CHEXPERT_CLASSES].fillna(0).values  # NaN → 0
 
    # Chỉ tính co-occur khi cả hai label == 1; uncertain (-1) → 0
    labels_bin = (labels == 1).astype(np.float32)       # {0, 1}
 
    n = len(CHEXPERT_CLASSES)
    adj  = np.zeros((n, n), dtype=np.float32)
    nums = np.zeros(n,      dtype=np.float32)
 
    for row in labels_bin:
        pos = np.where(row == 1)[0]
        nums[pos] += 1
        for i in pos:
            for j in pos:
                adj[i, j] += 1
 
    pickle.dump({'adj': adj, 'nums': nums}, open(ADJ_PATH, 'wb'))
    print(f'  Saved → {ADJ_PATH}  (shape {adj.shape})')
else:
    print(f'adj exists: {ADJ_PATH}')

adj exists: data/chexpert_adj.pkl


In [39]:
%%writefile  /kaggle/working/TULIP-MedML/src/configs/test.yaml

name: C1_smoke_test
seed: 42

data:
  root:      /kaggle/input/datasets/ashery/chexpert
  train_csv: /kaggle/working/TULIP-MedML/data/train_small.csv
  val_csv:   /kaggle/working/TULIP-MedML/data/valid.csv
  word_vec:  /kaggle/working/TULIP-MedML/data/chexpert_glove_word2vec.npy
  adj:       /kaggle/working/TULIP-MedML/data/chexpert_adj.pkl
  uncertain: keep
  img_size:  448

model:
  backbone:   resnet101
  pretrained: true
  gcn_in:     300
  t:          0.4

loss:
  type: bce

train:
  epochs:      20
  batch_size:  16
  lr:          0.01
  lrp:         0.1
  momentum:    0.9
  weight_decay: 0.0001
  epoch_step:  [30]
  workers:     4

output:
  save_dir: /kaggle/working/checkpoints/c1
  log_dir:  /kaggle/working/logs/c1

Overwriting /kaggle/working/TULIP-MedML/src/configs/test.yaml


In [41]:
!cd /kaggle/working/TULIP-MedML && python src/train.py \
    --config src/configs/test.yaml \
    --subset 100

Config: C1_smoke_test
[CheXpert] 44682 samples | policy=keep
[CheXpert] 234 samples | policy=keep
Subset mode: 100 train / 50 val
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/kaggle/working/TULIP-MedML/src/util.py:113: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To acce

In [42]:
!ls /kaggle/working/checkpoints/c1

checkpoint_epoch_0.pth.tar  checkpoint_epoch_9.pth.tar
checkpoint_epoch_1.pth.tar  model_best_0.3444.pth.tar
checkpoint_epoch_2.pth.tar  model_best_0.3452.pth.tar
checkpoint_epoch_3.pth.tar  model_best_0.3460.pth.tar
checkpoint_epoch_4.pth.tar  model_best_0.3468.pth.tar
checkpoint_epoch_5.pth.tar  model_best_0.3476.pth.tar
checkpoint_epoch_6.pth.tar  model_best_0.4094.pth.tar
checkpoint_epoch_7.pth.tar  model_best.pth.tar
checkpoint_epoch_8.pth.tar


In [38]:
# !cd /kaggle/working/TULIP-MedML && python src/train.py \
#     --config src/configs/c1.yaml 


Config: C1_baseline
[CheXpert] 44682 samples | policy=keep
[CheXpert] 234 samples | policy=keep
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/kaggle/working/TULIP-MedML/src/util.py:113: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use te